In [1]:
print('hi')

hi


In [2]:
%pwd

'c:\\Users\\VIVEK KUMAR\\Desktop\\medical chatbot\\Medical-Chatbot\\research'

In [3]:
import os 
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\VIVEK KUMAR\\Desktop\\medical chatbot\\Medical-Chatbot'

In [5]:
from langchain.document_loaders import PyPDFLoader ,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\Users\VIVEK KUMAR\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#extract test from pdf files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data, #path of the file
        glob="*.pdf", #to load only pdf files from data
        loader_cls=PyPDFLoader #we need to load it from  pypdfloader 
    ) 

    documents = loader.load()
    return documents

In [7]:
extracted_data = load_pdf_files('data')

In [8]:
len(extracted_data)

637

In [9]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata={'source': src}
            )
        )
    return minimal_docs    

In [10]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [11]:
minimal_docs[0]

Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='')

In [12]:
#split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    chunk_texts = text_splitter.split_documents(minimal_docs)
    return chunk_texts

In [13]:
text_splits = text_split(minimal_docs)
print(f" number of chunks: {len(text_splits)}")

 number of chunks: 5859


In [14]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        )
    return embeddings

embeddings = download_embeddings()


C:\Users\VIVEK KUMAR\AppData\Local\Temp\ipykernel_23768\174127969.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [15]:
vector = embeddings.embed_query("This is a sample document") # Display first 5 values of the embedding vector
print("vector length:", len(vector))

vector length: 384


In [16]:
from dotenv import load_dotenv
import os
load_dotenv()


True

In [17]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [18]:
from pinecone import Pinecone
pinecode_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecode_api_key)

In [19]:
pc

In [20]:
from pinecone import ServerlessSpec
index_name = "medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [21]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_splits,
    embedding=embeddings,
    index_name=index_name,
)


In [22]:
#load Existing index
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name,
)   

#add more data to the existing pinecone index

In [23]:
dswidth = Document(
    page_content="This is a sample document to test the Pinecone vector store.",
    metadata={"source": "test_document.pdf"}    
)

In [24]:
docsearch.add_documents([dswidth])
    

['29debf23-9644-40fc-aa5b-fa7dc40e3d22']

In [25]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [26]:
retriever_docs = retriever.invoke("what is diabetes?")
retriever_docs

[Document(id='f60a1f42-2cb2-476c-8c7b-ec27c2a24bdf', metadata={'source': 'data\\Medical_book.pdf'}, page_content='begin to fall. A person with diabetes mellitus either does\nnot make enough insulin, or makes insulin that does not\nwork properly. The result is blood sugar that remains\nhigh, a condition called hyperglycemia.\nDiabetes must be diagnosed as early as possible. If\nleft untreated, it can damage or cause failure of the eyes,\nkidneys, nerves, heart, blood vessels, and other body\norgans. Hypoglycemia, or low blood sugar, may also be\ndiscovered through blood sugar testing. Hypoglycemia is'),
 Document(id='0303ab86-2b98-4809-b124-b9c9fd542c1f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='begin to fall. A person with diabetes mellitus either does\nnot make enough insulin, or makes insulin that does not\nwork properly. The result is blood sugar that remains\nhigh, a condition called hyperglycemia.\nDiabetes must be diagnosed as early as possible. If\nleft untre

refine the output using llm

In [27]:
from langchain_openai import ChatOpenAI
chatModel = ChatOpenAI(model="gpt-4o") 

In [28]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [29]:
system_prompt = (
    "You are a helpful medical assistant. "
    "Use the following context to answer the question. "
    "If you don't know the answer, say you don't know.\n\n"
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [30]:
question_answering_chain = create_stuff_documents_chain(chatModel,prompt)
rag_chain = create_retrieval_chain(retriever,question_answering_chain)


In [ ]:
response = rag_chain.invoke({"input": "What are the common symptoms of hypertension?"})
print(response["answer"])